# Čtvrtletní pro forma výsledovka s procedurou PROC COMPUTAB

## Shrnutí pro vedení

Tento notebook sestavuje čtvrtletní pro forma výsledovku regionální banky přímo z měsíčních účetních dat pomocí procedury PROC COMPUTAB, tabulkové procedury pro tvorbu výkazů ze SAS/ETS. Úrokové výnosy, úrokové náklady, výnosy z poplatků a provozní náklady každého měsíce směruje do správného sloupce příslušného kalendářního čtvrtletí, poté pomocí programových bloků řádků počítá čistý úrokový výnos, zisk před zdaněním a čistý zisk a pomocí bloku sloupce sčítá čtyři čtvrtletí do celoročního součtu.

## Zdroje dat

Jediná syntetická datová sada `bank_ledger` simuluje jeden fiskální rok měsíčních položek výkazu pro středně velkou lokální banku. Dvanáct měsíčních pozorování je generováno inline pomocí `call streaminit`/`rand`, takže je notebook zcela soběstačný.

| Proměnná | Typ | Popis |
|----------|------|-------------|
| `stmtdate` | Num (DATE9.) | Datum výkazu ke konci měsíce (led–pro) |
| `loanint`  | Num | Úroky a poplatky z úvěrového portfolia (tis. USD) |
| `secint`   | Num | Úroky z portfolia investičních cenných papírů (tis. USD) |
| `depint`   | Num | Úroky placené z vkladů (tis. USD) |
| `borrint`  | Num | Úroky placené z výpůjček / úvěrů FHLB (tis. USD) |
| `feeinc`   | Num | Neúrokové výnosy (poplatky a servisní poplatky) (tis. USD) |
| `salaries` | Num | Náklady na mzdy a zaměstnanecké benefity (tis. USD) |
| `occupancy`| Num | Náklady na prostory a vybavení (tis. USD) |
| `othropex` | Num | Ostatní neúrokové provozní náklady (tis. USD) |
| `provision`| Num | Opravné položky ke krytí úvěrových ztrát (tis. USD) |
| `taxrate`  | Num | Efektivní daňová sazba aplikovaná na zisk před zdaněním |

# Čtvrtletní pro forma výsledovka s procedurou PROC COMPUTAB

Finanční týmy bank běžně převádějí měsíční hlavní knihu na **čtvrtletní výsledovku**, která ukazuje čistý úrokový výnos a výsledný čistý zisk. `PROC COMPUTAB` (SAS/ETS) je pro to přímo určena: rozvrhuje programovatelnou tabulku, jejíž **sloupce** jsou vykazovaná období a jejíž **řádky** jsou položky výkazu, a umožňuje počítat mezisoučty běžnou logikou DATA step v blocích řádků a sloupců.

V tomto notebooku:

1. Vygenerujeme jeden fiskální rok syntetických měsíčních účetních dat pro lokální banku.
2. Každý měsíc směrujeme do jeho kalendářního čtvrtletí a sestavíme čtvrtletní výsledovku.
3. V **bloku řádků** vypočteme čistý úrokový výnos, zisk před zdaněním a čistý zisk a v **bloku sloupce** sečteme čtvrtletí do celoročního součtu.
4. Znovu využijeme tabulku z `out=`, aby vypočtený výkaz mohl sloužit jako vstup pro navazující reporting.

## Krok 1 — Vygenerování syntetických měsíčních účetních dat

Simulujeme dvanáct pozorování ke konci měsíce. Úrokové výnosy z úvěrů a cenných papírů během roku mírně rostou, náklady na vklady a výpůjčky se odvíjejí od úrokového prostředí a výnosy z poplatků, provozní náklady a opravné položky ke krytí úvěrových ztrát obsahují realistický měsíční šum. `call streaminit` fixuje seed, takže jsou data reprodukovatelná.

In [1]:
data bank_ledger;
   CALL streaminit(20240115);
   FORMÁT stmtdate date9.;
   OPAKUJ month = 1 TO 12;
      /* Datum výkazu ke konci měsíce pro fiskální rok 2024 */
      stmtdate = mdy(month, 28, 2024);

      /* Mírný růstový trend během roku + měsíční šum */
      drift = 1 + 0.012 * (month - 1);

      /* Úrokové výnosy (tis. USD) */
      loanint = round(1850 * drift + rand('normal', 0, 45), 0.01);
      secint  = round( 420 * drift + rand('normal', 0, 15), 0.01);

      /* Úrokové náklady (tis. USD) */
      depint  = round( 540 * drift + rand('normal', 0, 22), 0.01);
      borrint = round( 130 * drift + rand('normal', 0, 10), 0.01);

      /* Neúrokové výnosy a náklady (tis. USD) */
      feeinc    = round(310 + rand('normal', 0, 18), 0.01);
      salaries  = round(720 + rand('normal', 0, 25), 0.01);
      occupancy = round(165 + rand('normal', 0, 8),  0.01);
      othropex  = round(240 + rand('normal', 0, 20), 0.01);

      /* Opravné položky ke krytí úvěrových ztrát, občas zvýšené */
      provision = round(95 + rand('exponential') * 40, 0.01);

      /* Efektivní daňová sazba */
      taxrate = 0.21;

      VÝSTUP;
   KONEC;
   PONECHAT stmtdate loanint secint depint borrint
        feeinc salaries occupancy othropex provision taxrate;
SPUSTIT;

PROCEDURA TISK data=bank_ledger noobs ŠTÍTEK;
   NÁZEV 'Syntetická měsíční hlavní kniha — lokální banka, FR 2024';
   ŠTÍTEK stmtdate='Datum výkazu'
          loanint='Úroky z úvěrů'
          secint='Úroky z cenných papírů'
          depint='Úroky z vkladů'
          borrint='Úroky z výpůjček'
          feeinc='Výnosy z poplatků'
          salaries='Mzdy a benefity'
          occupancy='Prostory a vybavení'
          othropex='Ostatní provozní náklady'
          provision='Opravné položky';
   FORMÁT loanint secint depint borrint feeinc
          salaries occupancy othropex provision comma8.2;
SPUSTIT;


                                Syntetická měsíční hlavní kniha — lokální banka, FR 2024                                

 Datum výkazu      Úroky z úvěrů      Úroky z cenných papírů    Úroky z vkladů      Úroky z výpůjček    Výnosy z poplatků  Mzdy a benefity   Prostory a vybavení     Ostatní provozní náklady    Opravné položky  taxrate
    28JAN2024           1,772.95                      423.79            531.47                128.99               339.33           699.38                171.95                       202.43             130.93     0.21
    28FEB2024           1,824.38                      421.13            564.85                125.53               294.09           767.29                162.79                       307.61             123.25     0.21
    28MAR2024           1,931.98                      442.06            536.64                131.71               295.72           724.03                153.26                       254.21             115.76     0.21
    28


NOTE: DATA bank_ledger


NOTE: Wrote bank_ledger (12 rows, 11 columns).
NOTE: DATA elapsed:
  wall  0.00 seconds
  cpu   0.00 seconds
NOTE: PROC PRINT data=bank_ledger

NOTE: PROC PRINT completed: 12 observations printed, 11 variables


## Krok 2 — Sestavení čtvrtletní výsledovky

Jádrem výkazu je níže uvedený krok `PROC COMPUTAB`.

- **`columns qtr1 qtr2 qtr3 qtr4 fy;`** definuje čtyři čtvrtletní sloupce plus celoroční sloupec.
- Neoznačený **vstupní blok** nastavuje automatickou proměnnou **`_col_`** hodnotou `qtr(stmtdate)`, která směruje každé měsíční pozorování do správného čtvrtletního sloupce. Protože je vstup ve výchozím nastavení transponován, každá účetní proměnná dopadne na řádek se stejným názvem.
- **Blok řádků** `inc_stmt:` běží jednou pro každý sloupec a počítá odvozené řádky — čistý úrokový výnos, celkové neúrokové náklady, zisk před zdaněním a čistý zisk — přesně tak, jak by to udělal účetní.
- **Blok sloupce** `total:` běží jednou pro každý řádek a sčítá čtyři čtvrtletí do sloupce `fy` (celý rok).

Příkazy `rows ... / ...` přidávají názvy, odsazení a čáry (`ol` nadčára, `ul` podčára, `dul` dvojitá podčára), takže výstup vypadá jako skutečný finanční výkaz.

In [2]:
NÁZEV 'Pro forma výsledovka — Community Bancorp, Inc.';
title2 'Fiskální rok 2024  (částky v tis. USD)';

PROCEDURA computab data=bank_ledger cspace=2 cwidth=11 out=qtr_income;

   /* Čtyři čtvrtletí plus sečtený celoroční sloupec */
   columns qtr1 qtr2 qtr3 qtr4 fy / FORMÁT=comma11.1;
   columns qtr1 / 'Q1';
   columns qtr2 / 'Q2';
   columns qtr3 / 'Q3';
   columns qtr4 / 'Q4';
   columns fy   / 'Celý' 'rok' +3;

   /* Řádky výsledovky, shora dolů */
   rows loanint  / 'Úroky a poplatky z úvěrů';
   rows secint   / 'Úroky z cenných papírů'         ul;
   rows intinc   / 'Celkové úrokové výnosy';
   rows depint   / 'Úroky z vkladů';
   rows borrint  / 'Úroky z výpůjček'               ul;
   rows intexp   / 'Celkové úrokové náklady';
   rows nii      / 'Čistý úrokový výnos'            ol skip;
   rows provision/ 'Opravné položky ke ztrátám'     ul;
   rows niiap    / 'Čistý úrok. výnos po OP'        skip;
   rows feeinc   / 'Neúrokové výnosy'               ul;
   rows salaries / 'Mzdy a benefity';
   rows occupancy/ 'Prostory a vybavení';
   rows othropex / 'Ostatní provozní náklady'       ul;
   rows nonintexp/ 'Celkové neúrokové náklady'      skip;
   rows pretax   / 'Zisk před zdaněním'             ol;
   rows taxexp   / 'Daň z příjmů'                   ul;
   rows netinc   / 'Čistý zisk'                     dul;

   /* Vstupní blok: směruj každý měsíc do jeho kalendářního čtvrtletí */
   _col_ = qtr(stmtdate);

   /* Blok řádků: počítej mezisoučty výkazu napříč všemi sloupci */
   inc_stmt:
      intinc    = loanint + secint;
      intexp    = depint + borrint;
      nii       = intinc - intexp;
      niiap     = nii - provision;
      nonintexp = salaries + occupancy + othropex;
      pretax    = niiap + feeinc - nonintexp;
      taxexp    = pretax * 0.21;
      netinc    = pretax - taxexp;

   /* Blok sloupce: sečti čtyři čtvrtletí do celoročního sloupce */
   TOTAL:
      fy = qtr1 + qtr2 + qtr3 + qtr4;
SPUSTIT;


                                     Pro forma výsledovka — Community Bancorp, Inc.                                     
                                         Fiskální rok 2024  (částky v tis. USD)                                         


                             The COMPUTAB Procedure                             

                             Q1           Q2           Q3           Q4         Celý  
                                                                                rok  
                           qtr1         qtr2         qtr3         qtr4           fy  
                    -----------  -----------  -----------  -----------  -----------  
  Úroky a poplatky z úvěrů
  loanint               5,529.3      5,818.7      5,963.5      6,277.0     23,588.4  
  Úroky z cenných papírů
  secint                1,287.0      1,334.9      1,342.0      1,452.9      5,416.8  
                    -----------  -----------  -----------  -----------  -----------  
  Celkové úrokové vý


NOTE: Option TITLE changed to Pro forma výsledovka — Community Bancorp, Inc..
NOTE: Option TITLE2 changed to Fiskální rok 2024  (částky v tis. USD).
NOTE: PROC COMPUTAB
NOTE: COMPUTAB OUT= dataset written to: qtr_income.csv
NOTE: PROC COMPUTAB statement used.


## Krok 3 — Opětovné využití výstupní datové sady procedury COMPUTAB

Výše uvedený krok `PROC COMPUTAB` zapsal svou vypočtenou tabulku do `qtr_income` pomocí `out=`. Každý řádek této datové sady je položka výkazu a každá sloupcová proměnná (`qtr1`–`qtr4`, `fy`) obsahuje vypočtenou hodnotu, takže může sloužit jako vstup pro navazující reporting. Níže vytiskneme sečtený celoroční sloupec, abychom potvrdili, že čísla souhlasí.

In [3]:
NÁZEV 'Výstupní datová sada COMPUTAB — celoroční sloupec';

PROCEDURA TISK data=qtr_income ŠTÍTEK noobs;
   PROMĚNNÁ _row_ fy;
   FORMÁT fy comma12.1;
   ŠTÍTEK _row_ = 'Položka' fy = 'Celý rok';
SPUSTIT;

NÁZEV;


                                   Výstupní datová sada COMPUTAB — celoroční sloupec                                    
                                         Fiskální rok 2024  (částky v tis. USD)                                         


  Položka   Celý rok
---------  ---------
loanint     23,588.4
secint       5,416.8
intinc      29,005.2
depint       6,831.2
borrint      1,650.7
intexp       8,482.0
nii         20,523.2
provision    1,568.9
niiap       18,954.3
feeinc       3,703.2
salaries     8,763.1
occupancy    1,985.6
othropex     2,944.8
nonintexp   13,693.5
pretax       8,964.1
taxexp       1,882.5
netinc       7,081.6




NOTE: Option TITLE changed to Výstupní datová sada COMPUTAB — celoroční sloupec.
NOTE: PROC PRINT data=qtr_income

NOTE: PROC PRINT completed: 17 observations printed, 2 variables


## Interpretace výsledků

Pro forma výkaz se čte shora dolů jako regulatorní výkaz banky (call report): celkové úrokové výnosy minus celkové úrokové náklady dávají **čistý úrokový výnos** — zde přibližně 20,5 milionu USD za rok — hlavní zdroj příjmů banky. Odečtení **opravných položek ke krytí úvěrových ztrát**, přičtení **výnosů z poplatků** a odečtení **neúrokových nákladů** dává zisk před zdaněním zhruba 9,0 milionu USD a aplikace 21% efektivní daňové sazby vytváří **čistý zisk** kolem 7,1 milionu USD. Blok sloupce `total:` přičítá čtyři čtvrtletí do celoročního sloupce, takže roční součty se z konstrukce shodují se součtem čtvrtletí — potvrzeno v datové sadě `out=`, kde celoroční `netinc` ve výši 7 081,6 odpovídá součtu čtyř čtvrtletních hodnot.

Protože se vše počítá uvnitř programovatelných bloků řádků a sloupců procedury `PROC COMPUTAB`, dosazení skutečné měsíční hlavní knihy nevyžaduje žádnou změnu logiky výkazu — mění se pouze vstupní datová sada. Datová sada `out=` pak zpřístupňuje vypočtený výkaz pro dashboardy, analýzu trendů nebo automatizaci podkladů pro představenstvo.